# Analyse du dédoublonnage

## Fonction d'analyse
Calcul et restitution du nombre de ligne en défut d'intégrité

In [1]:
from datetime import datetime
import json
from tab_dataset import Cdataset
import pandas as pd
import ntv_pandas as npd
import pathlib

def analyse_integrite(data, schema, affiche=True, indic=True):
    '''analyse les relations du DataFrame 'data' définies dans le schéma 'schema'.
    Le nombre de lignes en erreur par relation (dict) est retourné et optionnellement affiché (paramètre 'affiche=True') . 
    Les lignes en erreur sont optionnellement ajoutées (paramètre 'indic=True') à 'data' sous forme de champs booléens par relation.
    '''
    dic_errors = Cdataset(data).check_relationship(schema)
    dic_count = {name: len(errors) for name, errors in dic_errors.items()}
    if affiche:
        for name, total in dic_count.items():
            print('{:<50} {:>5}'.format(name, total))
    if indic:
        data['ok'] = True
        for name, errors in dic_errors.items():
            data[name] = True
            data.loc[errors, name] = False
            data['ok'] = data['ok'] & data[name] 
        if affiche:
            nb_ok = sum(data['ok'])
            nb_ko = len(data) - sum(data['ok'])      
            print("\nnombre d'enregistrements sans erreurs : ", nb_ok)
            print("nombre d'enregistrements avec au moins une erreur : ", nb_ko)
            print("dont doublons : ", dic_count['index - id_pdc_itinerance'])
            print("\ntaux d'erreur : ", round(nb_ko / len(data) * 100), ' %')
    return dic_count

## Schéma de données
Le schéma de données restreint à la propriété 'relationship' et construit à partir du modèle de données est le suivants :

In [2]:
# complément à inclure dans le schéma de données
schema = {
    'relationships': [
         # relation unicité des pdl
         {"fields": ["id_pdc_itinerance", "index"],                    "link" : "coupled" },   
         # relations inter entités
         {"fields": ["id_station_itinerance", "contact_operateur"],    "link" : "derived" },
         {"fields": ["id_station_itinerance", "nom_enseigne"],         "link" : "derived" },
         {"fields": ["id_station_itinerance", "coordonneesXY"],        "link" : "derived" },
         {"fields": ["id_pdc_itinerance", "id_station_itinerance"],    "link" : "derived" },
         # relations intra entité - station
         {"fields": ["id_station_itinerance", "nom_station"],          "link" : "derived" },
         {"fields": ["id_station_itinerance", "implantation_station"], "link" : "derived" },
         #{"fields": ["id_station_itinerance", "date_maj"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "nbre_pdc"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "condition_acces"],      "link" : "derived" },
         {"fields": ["id_station_itinerance", "horaires"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "station_deux_roues"],   "link" : "derived" },
         # relations intra entité - localisation
         {"fields": ["coordonneesXY", "adresse_station"],              "link" : "derived" }
    ]
}

## Initialisation des données
Fichier pandas

In [3]:
origine = 'datagouv_organization_or_owner'
priorite = 'priorite'
coord = 'coordonneesXY'
id_station = 'id_station_itinerance'
id_pdc = 'id_pdc_itinerance'
last_modif = 'last_modified'
date_maj = 'date_maj'
nom_station = 'nom_station'
adresse = 'adresse_station'
amenageur = 'nom_amenageur'
unit = 'unite'

unicite_stations = [id_station, origine, date_maj, last_modif]
filtre = [priorite,  date_maj, last_modif]
id_station_pdc = [id_station, id_pdc]
att_station = [id_station, date_maj, last_modif, amenageur, nom_station, coord]
att_pdc = [id_pdc, id_station, date_maj, last_modif, amenageur, nom_station, coord, adresse]
filtre_qualicharge = [nom_station, adresse, coord]

In [4]:
file_irve_brut = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260301.csv' # données brutes 09/03
file_irve = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260401.csv' # données dédoublonnées 09/03

irve_brut = pd.read_csv(file_irve_brut, sep=',', low_memory=False, dtype='object').reset_index()
irve_brut[last_modif] = irve_brut['datagouv_last_modified']

irve = pd.read_csv(file_irve, sep=',', low_memory=False, dtype='object').reset_index()
irve[last_modif] = irve['datagouv_last_modified']

## Données brutes

In [5]:
print('nombre de lignes : {}, nombre de pdc : {} \n'.format(len(irve_brut), len(irve_brut.groupby([id_pdc]).count())))
print(irve_brut.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_brut = analyse_integrite(irve_brut, schema)

nombre de lignes : 382238, nombre de pdc : 149243 

datagouv_organization_or_owner
QualiCharge                       60924
Engie Mobilités Electriques       58305
GIREVE                            32363
TotalEnergies Marketing France    29490
Mobilize Power Solutions          21033
IZIVIA                            15806
Driveco                           15551
ubitricity                        15349
Alizé                             14615
STATIONS-E                        13260
Name: index, dtype: int64 

index - id_pdc_itinerance                          295802
contact_operateur - id_station_itinerance          75149
nom_enseigne - id_station_itinerance               57990
coordonneesXY - id_station_itinerance              118195
id_station_itinerance - id_pdc_itinerance          150278
nom_station - id_station_itinerance                57585
implantation_station - id_station_itinerance       79117
nbre_pdc - id_station_itinerance                   73318
condition_acces - id_station_i

## Dédoublonnage actuel

In [6]:
print('nombre de lignes : {} \n'.format(len(irve)))
print(irve.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_actuel = analyse_integrite(irve, schema)

nombre de lignes : 149705 

datagouv_organization_or_owner
QualiCharge                                  46861
GIREVE                                       32363
Alizé                                        13905
IZIVIA                                       12985
Eco-Movement                                 10929
Indigo Group                                  6923
Driveco                                       3935
TotalEnergies Marketing France                1849
Engie Mobilités Electriques                   1827
Citeos Ingénierie IdF & Est (Cogelum IdF)     1485
Name: index, dtype: int64 

index - id_pdc_itinerance                            820
contact_operateur - id_station_itinerance           1834
nom_enseigne - id_station_itinerance                4140
coordonneesXY - id_station_itinerance               5515
id_station_itinerance - id_pdc_itinerance             28
nom_station - id_station_itinerance                 2462
implantation_station - id_station_itinerance        2465
nbre

## Dédoublonnage proposé
Les critères utilisés pour éliminer les doublons sont par ordre : priorite (datagouv_organization_or_owner), date (date_maj, datagouv_last_modified)

In [7]:
# dédoublonnage des stations (suivant critères d'unicité)
stations = irve_brut.drop_duplicates(unicite_stations).copy()

print("nombre de stations brutes {}, stations suivant critère d'unicité {} et stations uniques {} ".format(len(irve_brut), len(stations), len(irve_brut.drop_duplicates(id_station))))

nombre de stations brutes 382238, stations suivant critère d'unicité 127611 et stations uniques 59831 


### Dédoublonnage direct station
A l'issue de cette étape, on a une liste de stations uniques respectant les critères de filtrage

avec l'origine la plus prioritaire et la mise à jour la plus récente

In [8]:
# choix du critère de priorité (Qualicharge)
stations[priorite] = stations[origine] == 'QualiCharge'

# dédoublonnage des stations suivant son id_station_itinerance avec filtrage suivant les critères retenus
stat_direct = stations.sort_values(by=[id_station] + filtre).drop_duplicates(id_station, keep='last').copy()
if stations[priorite].sum() != stat_direct[priorite].sum():
    print('priorité non respectée')

In [9]:
print('nombre de stations initiales {}, stations dédoublonnées {} (supprimées {})'.format(len(stations), len(stat_direct), len(stations) - len(stat_direct)))
stat_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

nombre de stations initiales 127611, stations dédoublonnées 59831 (supprimées 67780)


datagouv_organization_or_owner
QualiCharge                                  13661
Eco-Movement                                 10599
GIREVE                                       10080
IZIVIA                                        6665
Alizé                                         4564
GREENEA                                       1910
Load Stations                                 1107
Driveco                                        892
Syndicat Départemental d'Energie du Tarn       700
ZE-WATT                                        624
Citeos Ingénierie IdF & Est (Cogelum IdF)      616
SOREGIES                                       451
Electric 55 Charging                           394
ZEborne                                        377
Mobilize Power Solutions                       373
Name: index, dtype: int64

### Dédoublonnage direct pdc
A l'issue de cette étape, chaque pdc est présent une seule fois sur la station respectant les critères de filtrage

In [10]:
# dédoublonnage des pdc présents sur plusieurs stations
pdc_stat = stat_direct[unicite_stations + [priorite]].merge(irve_brut, how='left', on=unicite_stations)
pdc_stat_unique = pdc_stat.sort_values(by=id_station_pdc + filtre).drop_duplicates(id_station_pdc, keep='last').copy()

# dédoublonnage des pdc suivant son id_pdc avec filtrage par priorite, date_maj, last_modif (priorité déja filtrée sur les stations mais nécessaire)
pdc_direct =  pdc_stat_unique.sort_values(by=[id_pdc] + filtre).drop_duplicates(id_pdc, keep='last').copy()
del(pdc_direct['index'])
pdc_direct = pdc_direct.reset_index()

### Bilan dédoublonnage direct

In [11]:
print("nombre de pdc initial {}, avec dédoublonnage des pdc multi-stations {} et avec dédoublonnage de l'historique {} (supprimés {})\n".format(len(pdc_stat), len(pdc_stat_unique), len(pdc_direct), len(pdc_stat)-len(pdc_direct)))
print(pdc_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_direct = analyse_integrite(pdc_direct, schema)

nombre de pdc initial 175130, avec dédoublonnage des pdc multi-stations 174555 et avec dédoublonnage de l'historique 146126 (supprimés 29004)

datagouv_organization_or_owner
QualiCharge                       60924
GIREVE                            24053
Alizé                             13853
IZIVIA                            12262
Indigo Group                       6923
Eco-Movement                       4009
Driveco                            3460
TotalEnergies Marketing France     1848
Engie Mobilités Electriques        1745
Electric 55 Charging               1300
Name: index, dtype: int64 

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              2
nom_enseigne - id_station_itinerance                   7
coordonneesXY - id_station_itinerance                998
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                 1265
implantation_station - id_station_itinerance          71

### Dédoublonnage indirect des stations

In [12]:
stations_direct = pdc_direct.drop_duplicates([id_station]).copy()

In [13]:
# Cette fonction donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec le champ 'attribut' identique et venant de plusieurs origines
def indirect_station_ext(stations, attribut, affiche=True):
    dupl_att_origine = ~stations.duplicated(keep=False, subset=[attribut, origine])
    
    stations_ext = stations[dupl_att_origine].copy()
    stations_int = stations[~dupl_att_origine].copy()

    filtrage = stations_ext.sort_values(by=[attribut, priorite])
    stat_att = filtrage.drop_duplicates([attribut], keep='last').copy()
    
    duplicates = filtrage.duplicated(subset=[attribut], keep='last')
    #print(duplicates)
    duplicated = stations_ext.loc[duplicates] #.copy()
    
    resultat = pd.concat([stations_int, stat_att])
    if affiche :
        print("nombre de stations dupliquées pour l'attribut '{:<15}' : {}".format(attribut, len(stations_ext) - len(stat_att)))
        print("nombre initial de stations {}, avec dédoublonnage {} {}\n".format(len(stations), attribut, len(resultat)))
    return (resultat, duplicated)

#### Evaluation des stations avec un attribut identique et origine différente
Cette étape donne la liste des stations en éliminant (suivant les critères de filtrage) celles avec un attribut identiques et venant de plusieurs origines

In [14]:
stations_xy, dupl_xy = indirect_station_ext(stations_direct, coord)
stations_nom, dupl_nom = indirect_station_ext(stations_direct, nom_station)
stations_adr, dupl_adr = indirect_station_ext(stations_direct, adresse)

nombre de stations dupliquées pour l'attribut 'coordonneesXY  ' : 950
nombre initial de stations 45681, avec dédoublonnage coordonneesXY 44731

nombre de stations dupliquées pour l'attribut 'nom_station    ' : 197
nombre initial de stations 45681, avec dédoublonnage nom_station 45484

nombre de stations dupliquées pour l'attribut 'adresse_station' : 104
nombre initial de stations 45681, avec dédoublonnage adresse_station 45577



#### Dédoublonnage des stations hors Qualicharge

In [15]:
stations_xy, dupl_xy = indirect_station_ext(stations_direct, coord)

nombre de stations dupliquées pour l'attribut 'coordonneesXY  ' : 950
nombre initial de stations 45681, avec dédoublonnage coordonneesXY 44731



In [16]:
stations_nom, dupl_nom = indirect_station_ext(stations_xy, nom_station)

nombre de stations dupliquées pour l'attribut 'nom_station    ' : 68
nombre initial de stations 44731, avec dédoublonnage nom_station 44663



In [17]:
stations_adr, dupl_adr = indirect_station_ext(stations_nom, adresse)

nombre de stations dupliquées pour l'attribut 'adresse_station' : 45
nombre initial de stations 44663, avec dédoublonnage adresse_station 44618



In [18]:
# le gain avec la suppression des doublons d'adresse est faible et on supprime quelques stations différentes, on se limite aux coordonnées et au nom
stations_indirect = stations_nom

In [19]:
stations_indirect.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

datagouv_organization_or_owner
QualiCharge                                  13661
GIREVE                                        6684
IZIVIA                                        6621
Alizé                                         4242
Eco-Movement                                  3921
Driveco                                        889
ZE-WATT                                        624
Citeos Ingénierie IdF & Est (Cogelum IdF)      615
Syndicat Départemental d'Energie du Tarn       601
SOREGIES                                       396
Electric 55 Charging                           394
ZEborne                                        377
Mobilize Power Solutions                       347
Rossini Energy                                 311
Engie Mobilités Electriques                    309
Name: index, dtype: int64

#### Dédoublonnage des stations Qualicharge
origine Qualicharge, même nom, même coordonnées, même adresse, unité différente

In [20]:
# Cette fonction donne la liste des stations en éliminant (suivant les critères de filtrage) les stations Qualicharge avec les mêmes champs nom, coordonnées et adresse mais des unités différentes
def indirect_qualicharge(stations, affiche=True):
    stat = stations.copy()
    stat[unit] = stat[id_station].str[:5]
    #filtre_qualicharge = [nom_station, adresse, coord]
    
    stat_quali = stat[stat[origine]=='QualiCharge'].copy()
    stat_not_quali = stat[~(stat[origine]=='QualiCharge')].copy()
    
    stat_quali['unique'] = ~stat_quali.duplicated(filtre_qualicharge + [unit])
    filtrage = stat_quali.sort_values(by=['unique'] + filtre_qualicharge + [date_maj])
    stat_quali['doublon'] = filtrage.duplicated(subset=filtre_qualicharge, keep='last')
    
    duplicated = stat_quali[stat_quali['doublon']]
    stat_result = stat_quali[~stat_quali['doublon']]
    stat_result.drop(['doublon', 'unique'], axis=1)

    resultat = pd.concat([stat_not_quali, stat_result])
    resultat.drop(['unite'], axis=1)
    
    if affiche :
        print("nombre de stations dupliquées entre unités : {}".format(len(duplicated)))
        print("nombre initial de stations {}, avec dédoublonnage {}\n".format(len(stations), len(resultat)))
    return (resultat, duplicated)

In [21]:
stat_indirect_quali, dupl_quali = indirect_qualicharge(stations_indirect)

nombre de stations dupliquées entre unités : 1696
nombre initial de stations 44663, avec dédoublonnage 42967



### Dédoublonnage indirect pdc

In [22]:
pdc_indirect = stat_indirect_quali[[id_station]].merge(pdc_direct, how='left', on=id_station)

In [23]:
print("nombre de pdc initial {} et avec dédoublonnage {} (supprimés {})\n".format(len(pdc_direct), len(pdc_indirect), len(pdc_direct)-len(pdc_indirect)))
print(pdc_indirect.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15], '\n')
res_indirect = analyse_integrite(pdc_indirect, schema)

nombre de pdc initial 146126 et avec dédoublonnage 138904 (supprimés 7222)

datagouv_organization_or_owner
QualiCharge                                  56565
GIREVE                                       21698
Alizé                                        13826
IZIVIA                                       12262
Indigo Group                                  6913
Eco-Movement                                  4008
Driveco                                       3460
TotalEnergies Marketing France                1845
Engie Mobilités Electriques                   1743
Electric 55 Charging                          1300
Qovoltis                                      1198
Citeos Ingénierie IdF & Est (Cogelum IdF)     1118
Lidl                                          1077
e-Totem                                        896
Rossini Energy                                 855
Name: index, dtype: int64 

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance  

## Synthèse des dédoublonnages

In [24]:
print('données brutes       : total pdc {:<7}'.format(len(irve_brut.groupby([id_pdc]).count())))
print('solution actuelle    : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(irve), sum(irve['ok']), len(irve) - sum(irve['ok']), res_actuel['index - id_pdc_itinerance']))
print('proposition direct   : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(pdc_direct), sum(pdc_direct['ok']), len(pdc_direct) - sum(pdc_direct['ok']), res_direct['index - id_pdc_itinerance']))
print('proposition indirect : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(pdc_indirect), sum(pdc_indirect['ok']), len(pdc_indirect) - sum(pdc_indirect['ok']), res_indirect['index - id_pdc_itinerance']))


données brutes       : total pdc 149243 
solution actuelle    : total pdc 149705  avec pdc ok 133735 , pdc avec erreur 15970 dont doublons 820
proposition direct   : total pdc 146126  avec pdc ok 135867 , pdc avec erreur 10259 dont doublons 0
proposition indirect : total pdc 138904  avec pdc ok 134679 , pdc avec erreur  4225 dont doublons 0


## Test des cas de doublons identifiés

- cas 0 : doublon de pdc d'origine différentes sur une même station
- cas 1 : pdc sur deux stations avec identifiants de station différents
- cas 2 : station avec deux origines, des identifiants différents et mêmes coordonnées
- cas 3 : station avec deux origines, des identifiants et des coordonnées différents, des noms identiques
- cas 4 : station avec deux origines, des identifiants et des coordonnées et des noms différents, des adresses identiques
- cas 5 : station Qualicharge décommissionnée partiellement
- cas 6 : station Qualicharge décommissionnée totalement
- cas 7 : station Qualicharge avec changement d'unité d'exploitation

In [25]:
def test_doublons(irve):
    return {
        'station Tesla de 48 pdc (cas 0)': len(irve[irve[id_station]=='FRTSLP16281'])==48,
        'station Atlante de 16 pdc (cas 1)': (len(irve[irve[id_station]=='FRATLP1136899124034716498']) + len(irve[irve[id_station]=='FRATLPFR01092'])) == 16,
        'pdc Ionity sur deux stations (cas 1)': len(irve[irve[id_pdc]=='FRIOYE423408']) == 1,                 
        'stations de 2 pdc (cas 2)': len(irve[irve[coord]=='[-0.00097, 49.32456]'])==2,
        'stations de 12 pdc (cas 2)': len(irve[irve[coord]=='[-0.05677, 48.72293]'])==12,
        'stations de 2 pdc (cas 3)': len(irve[irve[nom_station]=='VALENCE EN POITOU_SALLE DES FETES'])==2,
        'stations de 3 pdc (cas 4)': len(irve[irve[adresse]=='15 Av. Président Georges Pompidou'])==3,
        'station R3 de 1 pdc (cas 5)': len(irve[irve[id_station]=='FRR3MP1063577'])==1,
        'station Total de x pdc (cas 7)': len(irve[irve[coord]=='[5.40266, 43.26239]'])==2
    }
def cumul_tests(resultat):
    return sum(resultat.values())

In [26]:
res_actuel = test_doublons(irve)
propos_direct = test_doublons(pdc_direct)
propos_indirect = test_doublons(pdc_indirect)
sum(res_actuel.values()), sum(propos_direct.values()), sum(propos_indirect.values())

(2, 4, 8)

In [27]:
res_actuel

{'station Tesla de 48 pdc (cas 0)': True,
 'station Atlante de 16 pdc (cas 1)': False,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': False,
 'stations de 12 pdc (cas 2)': False,
 'stations de 2 pdc (cas 3)': False,
 'stations de 3 pdc (cas 4)': False,
 'station R3 de 1 pdc (cas 5)': False,
 'station Total de x pdc (cas 7)': False}

In [28]:
propos_indirect

{'station Tesla de 48 pdc (cas 0)': True,
 'station Atlante de 16 pdc (cas 1)': True,
 'pdc Ionity sur deux stations (cas 1)': True,
 'stations de 2 pdc (cas 2)': True,
 'stations de 12 pdc (cas 2)': True,
 'stations de 2 pdc (cas 3)': True,
 'stations de 3 pdc (cas 4)': False,
 'station R3 de 1 pdc (cas 5)': True,
 'station Total de x pdc (cas 7)': True}

In [29]:
irve[irve[coord]=='[5.40266, 43.26239]'][att_pdc]

,id_pdc_itinerance,id_station_itinerance,date_maj,last_modified,nom_amenageur,nom_station,coordonneesXY,adresse_station
143754,FRTCBE008442,FRTCBP01910,2025-12-21,2026-03-08T18:22:04.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE"
144035,FRTCBE008441,FRTCBP01910,2025-12-21,2026-03-08T18:22:04.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE"
148079,FRHXWE008441,FRHXWP01910,2026-01-07,2026-03-08T18:22:04.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE"
149191,FRHXWE008442,FRHXWP01910,2026-01-07,2026-03-08T18:22:04.000000+0000,TotalEnergies Charging Services,AMP | 1 Rue Mignard,"[5.40266, 43.26239]","1 Rue Mignard, 13009 MARSEILLE"
